# CS 5326 PA1 - Training on Kaggle

Clones the assignment implementation, tokenizes the attached TinyStories dataset with the assignment's fixed tokenizer, runs the full 10,000-step training run with the fixed architecture and the recommended hyperparameters, and exports `final_model.pt`.

In [ ]:
!pip install -q einops jaxtyping tokenizers huggingface-hub wrapt tqdm
!rm -rf cs5326-pa1
!git clone https://github.com/syeddaniyalg/cs5326-pa1.git
%cd cs5326-pa1

## Configuration

Every path, architecture dimension, and training hyperparameter used below is defined here once, as constants, and referenced by name everywhere else in this notebook.

In [ ]:
HF_TOKENIZER_REPO = "alooboii/pa1-tinystories"
HF_TOKENIZER_FILENAME = "tokenizer/tokenizer.json"

KAGGLE_INPUT_TRAIN_TEXT = "/kaggle/input/tiny-stories-ds/TinyStoriesV3-GPT4-train.txt"
KAGGLE_INPUT_VAL_TEXT = "/kaggle/input/tiny-stories-ds/TinyStoriesV3-GPT4-valid.txt"

TRAIN_PATH = "data/tinystories/train.bin"
VAL_PATH = "data/tinystories/validation.bin"
CHECKPOINT_PATH = "/kaggle/working/checkpoint.pt"
FINAL_MODEL_PATH = "/kaggle/working/final_model.pt"

VOCAB_SIZE = 8192
CONTEXT_LENGTH = 256
D_MODEL = 512
NUM_LAYERS = 4
N_Q_HEADS = 16
N_KV_HEADS = 4
D_FF = 1344
ROPE_THETA = 10000.0
NORM_EPS = 1e-5

BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 16
NUM_STEPS = 10000

LEARNING_RATE_MAX = 3e-4
LEARNING_RATE_MIN = 3e-5
WARMUP_STEPS = 200
COSINE_STEPS = 9999

BETA1 = 0.9
BETA2 = 0.95
ADAM_EPS = 1e-8
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0

EVAL_INTERVAL = 200
LOG_INTERVAL = 50
CHECKPOINT_INTERVAL = 500
NUM_VALIDATION_BATCHES = 20

SEED = 42
DEVICE = "cuda"

import os
from pathlib import Path

import numpy as np
from tokenizers import Tokenizer
from tqdm import tqdm

tokenizer = Tokenizer.from_file(tokenizer_path)
eot_id = tokenizer.token_to_id("<|endoftext|>")

Path(TRAIN_PATH).parent.mkdir(parents=True, exist_ok=True)

DELIMITER = "<|endoftext|>"
CHUNK_CHARS = 32_000_000
ENCODE_BATCH_SIZE = 5000


def tokenize_to_bin(text_path, out_path):
    def flush(stories, out_file):
        if not stories:
            return
        for encoding in tokenizer.encode_batch(stories, add_special_tokens=False):
            ids = np.array(encoding.ids + [eot_id], dtype="<u2")
            out_file.write(ids.tobytes())

    total_bytes = os.path.getsize(text_path)
    buffer = ""
    batch = []
    with open(text_path, "r", encoding="utf-8") as in_file, open(out_path, "wb") as out_file:
        with tqdm(total=total_bytes, unit="B", unit_scale=True, desc=Path(text_path).name, mininterval=1.0) as progress:
            while True:
                chunk = in_file.read(CHUNK_CHARS)
                if not chunk:
                    break
                buffer += chunk
                parts = buffer.split(DELIMITER)
                buffer = parts.pop()
                for story in parts:
                    if story.strip():
                        batch.append(story)
                    if len(batch) >= ENCODE_BATCH_SIZE:
                        flush(batch, out_file)
                        batch = []
                progress.n = in_file.buffer.tell()
                progress.refresh()
            if buffer.strip():
                batch.append(buffer)
            flush(batch, out_file)
            progress.n = total_bytes
            progress.refresh()


tokenize_to_bin(KAGGLE_INPUT_TRAIN_TEXT, TRAIN_PATH)
tokenize_to_bin(KAGGLE_INPUT_VAL_TEXT, VAL_PATH)

In [ ]:
from huggingface_hub import hf_hub_download

tokenizer_path = hf_hub_download(
    repo_id=HF_TOKENIZER_REPO,
    repo_type="dataset",
    filename=HF_TOKENIZER_FILENAME,
)

In [ ]:
from pathlib import Path

import numpy as np
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(tokenizer_path)
eot_id = tokenizer.token_to_id("<|endoftext|>")

Path(TRAIN_PATH).parent.mkdir(parents=True, exist_ok=True)


def tokenize_to_bin(text_path, out_path, batch_size=10000):
    stories = Path(text_path).read_text(encoding="utf-8").split("<|endoftext|>")
    with open(out_path, "wb") as out_file:
        for start in range(0, len(stories), batch_size):
            batch = [story for story in stories[start : start + batch_size] if story.strip()]
            if not batch:
                continue
            for encoding in tokenizer.encode_batch(batch, add_special_tokens=False):
                ids = np.array(encoding.ids + [eot_id], dtype="<u2")
                out_file.write(ids.tobytes())


tokenize_to_bin(KAGGLE_INPUT_TRAIN_TEXT, TRAIN_PATH)
tokenize_to_bin(KAGGLE_INPUT_VAL_TEXT, VAL_PATH)

## Train

Fixed architecture from the assignment manual (19,272,192 parameters) and the recommended starting hyperparameters (Section 7.1), all defined as constants above and passed through here.

In [ ]:
!python -m src.train \
  --train_path {TRAIN_PATH} \
  --val_path {VAL_PATH} \
  --checkpoint_path {CHECKPOINT_PATH} \
  --vocab_size {VOCAB_SIZE} \
  --context_length {CONTEXT_LENGTH} \
  --d_model {D_MODEL} \
  --num_layers {NUM_LAYERS} \
  --n_q_heads {N_Q_HEADS} \
  --n_kv_heads {N_KV_HEADS} \
  --d_ff {D_FF} \
  --rope_theta {ROPE_THETA} \
  --norm_eps {NORM_EPS} \
  --batch_size {BATCH_SIZE} \
  --gradient_accumulation_steps {GRADIENT_ACCUMULATION_STEPS} \
  --num_steps {NUM_STEPS} \
  --learning_rate_max {LEARNING_RATE_MAX} \
  --learning_rate_min {LEARNING_RATE_MIN} \
  --warmup_steps {WARMUP_STEPS} \
  --cosine_steps {COSINE_STEPS} \
  --beta1 {BETA1} \
  --beta2 {BETA2} \
  --adam_eps {ADAM_EPS} \
  --weight_decay {WEIGHT_DECAY} \
  --max_grad_norm {MAX_GRAD_NORM} \
  --eval_interval {EVAL_INTERVAL} \
  --log_interval {LOG_INTERVAL} \
  --checkpoint_interval {CHECKPOINT_INTERVAL} \
  --num_validation_batches {NUM_VALIDATION_BATCHES} \
  --seed {SEED} \
  --device {DEVICE}

## Export the submission artifact

Model-only state dict, CPU, float16, per Section 7.4.

In [ ]:
import torch

from src.data import load_checkpoint
from src.model import TransformerLM
from src.optim import AdamW

model = TransformerLM(VOCAB_SIZE, CONTEXT_LENGTH, D_MODEL, NUM_LAYERS, N_Q_HEADS, N_KV_HEADS, D_FF, ROPE_THETA, norm_eps=NORM_EPS, device=DEVICE)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE_MAX, betas=(BETA1, BETA2), eps=ADAM_EPS, weight_decay=WEIGHT_DECAY)
train_generator = torch.Generator()
val_generator = torch.Generator()
load_checkpoint(CHECKPOINT_PATH, model, optimizer, train_generator, val_generator)

state = {
    name: tensor.detach().cpu().to(torch.float16) if tensor.is_floating_point() else tensor.detach().cpu()
    for name, tensor in model.state_dict().items()
}
torch.save(state, FINAL_MODEL_PATH)

## Sample generations

A quick look at the trained model under a couple of decoding settings.

In [ ]:
from src.generate import generate

prompt_ids = torch.tensor(tokenizer.encode("Once upon a time", add_special_tokens=False).ids, device=DEVICE)

for temperature, top_p in [(1.0, 1.0), (0.8, 0.9)]:
    generated = generate(model, prompt_ids, 200, model.context_length, temperature=temperature, top_p=top_p, eot_token_id=eot_id)
    text = tokenizer.decode(generated.tolist(), skip_special_tokens=False)
    print(f"temperature={temperature} top_p={top_p}")
    print(text)
    print()